In [1]:
# Import libraries
from openai import OpenAI
import httpx
import os
from dotenv import load_dotenv
import arxiv
import numpy as np

In [2]:
# Load the environment variables
load_dotenv()

True

In [13]:
# Get the openai and jina 
JINA_API_KEY = os.getenv("JINA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_EMBEDDING_API_KEY")

In [4]:
# Initialize the OpenAI client
openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [6]:
# Get arXiv paper metadata
search = arxiv.Search(
    query="cat:cs.* OR cat:stat.ML",
    max_results=100, 
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending
)

client = arxiv.Client(
    page_size=100,
    delay_seconds=3.0,
    num_retries=3
)

# Fetch results
results = client.results(search)

# Test with just the FIRST paper
first_paper = next(results)

print("Title:       ", first_paper.title)
print("Authors:     ", [a.name for a in first_paper.authors])
print("Published:   ", first_paper.published)
print("Category:    ", first_paper.primary_category)
print("All Categories:", first_paper.categories)
print("Abstract:    ", first_paper.summary[:300], "...")  # first 300 chars
print("PDF URL:     ", first_paper.pdf_url)
print("ArXiv ID:    ", first_paper.entry_id)

Title:        Scale Space Diffusion
Authors:      ['Soumik Mukhopadhyay', 'Prateksha Udhayanan', 'Abhinav Shrivastava']
Published:    2026-03-09 17:59:42+00:00
Category:     cs.CV
All Categories: ['cs.CV', 'cs.AI']
Abstract:     Diffusion models degrade images through noise, and reversing this process reveals an information hierarchy across timesteps. Scale-space theory exhibits a similar hierarchy via low-pass filtering. We formalize this connection and show that highly noisy diffusion states contain no more information th ...
PDF URL:      https://arxiv.org/pdf/2603.08709v1
ArXiv ID:     http://arxiv.org/abs/2603.08709v1


In [7]:
text_to_embed = f"{first_paper.title}. {first_paper.summary}"

In [8]:
text_to_embed

'Scale Space Diffusion. Diffusion models degrade images through noise, and reversing this process reveals an information hierarchy across timesteps. Scale-space theory exhibits a similar hierarchy via low-pass filtering. We formalize this connection and show that highly noisy diffusion states contain no more information than small, downsampled images - raising the question of why they must be processed at full resolution. To address this, we fuse scale spaces into the diffusion process by formulating a family of diffusion models with generalized linear degradations and practical implementations. Using downsampling as the degradation yields our proposed Scale Space Diffusion. To support Scale Space Diffusion, we introduce Flexi-UNet, a UNet variant that performs resolution-preserving and resolution-increasing denoising using only the necessary parts of the network. We evaluate our framework on CelebA and ImageNet and analyze its scaling behavior across resolutions and network depths. Ou

In [ ]:
# Jina AI Test
print("\n[1] Testing Jina AI Embeddings (jina-embeddings-v3)")
jina_tokens = 0

# Text to embed
text_to_embed = f"{first_paper.title}. {first_paper.summary}"

try:
    headers = {
        "Authorization": f"Bearer {JINA_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": "jina-embeddings-v3",
        "normalized": True,
        "embedding_type": "float",
        "input": [text_to_embed],
    }
    with httpx.Client(timeout=30.0) as client:
        response = client.post("https://api.jina.ai/v1/embeddings", json=payload, headers=headers)
        response.raise_for_status()
        jina_data = response.json()
        
        jina_vector = jina_data["data"][0]["embedding"]
        jina_tokens = jina_data["usage"]["total_tokens"]
        
        print(f"✅ Success! Generated vector with dimension: {len(jina_vector)}")
        print(f"📊 Tokens consumed: {jina_tokens}")
        print("-> Check your Jina AI dashboard to see this reflected.")
        
except Exception as e:
    print(f"❌ Jina Error: {e}")


[1] Testing Jina AI Embeddings (jina-embeddings-v3)
✅ Success! Generated vector with dimension: 1024
📊 Tokens consumed: 264
-> Check your Jina AI dashboard to see this reflected.


In [15]:
# Check the vector
print(jina_vector)

[0.06591721, -0.05210009, -0.0411294, -0.05112611, 0.01199377, 0.14988586, -0.12830229, 0.05340024, -0.06932539, -0.03677953, -0.15499628, 0.10609687, 0.02295298, -0.08979473, 0.0020034, -0.03038264, -0.09671227, 0.07100572, 0.0539941, -0.08732505, 0.05573552, 0.05198995, -0.0315701, 0.10325383, -0.02196145, 0.20248544, -0.08519106, -0.05178087, -0.09632409, 0.05575946, 0.13734947, -0.02059085, 0.0369673, -0.022861, -0.02250141, -0.01216793, -0.04590992, -0.00251909, -0.08177055, -0.0090038, 0.02803827, 0.02115838, -0.0055938, -0.0315569, 0.02417659, 0.09558706, 0.04686808, 0.00561561, -0.02545765, 0.05191743, -0.00755592, 0.02916424, -0.00444883, 0.04894495, 0.06469339, -0.04967536, 0.03185447, 0.00566744, 0.03954676, 0.03766168, -0.04126991, 0.00219288, 0.02656505, 0.03186165, 0.00665602, -0.00653723, 0.05299975, -0.00085511, -0.04124529, -0.03919948, -0.05436949, 0.03995049, -0.01628879, 0.02784718, 0.02766308, -0.0335686, 0.10096865, 0.00353687, -0.02996258, 0.01784197, 0.07448138,

In [16]:
# OpenAI Test
print("\n[2] Testing OpenAI Embeddings (text-embedding-3-small)")
openai_tokens = 0
try:
    response = openai_client.embeddings.create(
        input=[text_to_embed],
        model="text-embedding-3-small"
    )
    
    openai_vector = response.data[0].embedding
    openai_tokens = response.usage.total_tokens
    
    print(f"✅ Success! Generated vector with dimension: {len(openai_vector)}")
    print(f"📊 Tokens consumed: {openai_tokens}")
    print("-> Check your OpenAI platform dashboard to see this reflected (Usage page).")
    
except Exception as e:
    print(f"❌ OpenAI Error: {e}")


[2] Testing OpenAI Embeddings (text-embedding-3-small)
✅ Success! Generated vector with dimension: 1536
📊 Tokens consumed: 205
-> Check your OpenAI platform dashboard to see this reflected (Usage page).


In [17]:
# Check the vector
print(openai_vector)

[0.030784526839852333, 0.004614472389221191, 0.03027145192027092, -0.017957640811800957, -0.006945759057998657, -0.0023376999888569117, 0.005265437066555023, -0.0016835288843140006, -0.07029134035110474, 0.0427904948592186, -0.012294570915400982, 0.002390610985457897, -0.06141513213515282, 0.02098478563129902, 0.018150044605135918, -0.003168240888044238, 0.0468950979411602, 0.020997613668441772, 0.009825395420193672, -0.0021469001658260822, 0.024204334244132042, -0.03342686593532562, 0.004104603547602892, -0.006317241583019495, -0.01570010930299759, -0.03263159841299057, 0.04786993935704231, 0.06085075065493584, 0.04402187466621399, 0.027321267873048782, -0.009722779504954815, -0.021767226979136467, -0.025102216750383377, -0.010934920981526375, 0.003751864191144705, 0.01051163300871849, -0.015879685059189796, 0.07588385790586472, -0.042841799557209015, 0.017200855538249016, 0.025089390575885773, 0.0018390548648312688, -0.024550661444664, 0.0355561301112175, 0.016854528337717056, -0.004

In [9]:
def get_embedding(text):
    response = openai_client.embeddings.create(
        input=[text],
        model="text-embedding-3-small"
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

# ---- Your documents (the "database") ----
documents = [
    text_to_embed,
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language for data science.",
    "Machine learning is a subset of artificial intelligence.",
    "The Great Wall of China is one of the seven wonders.",
    "Neural networks are inspired by the human brain.",
]

# ---- Embed all documents ----
print("📦 Embedding documents...")
doc_embeddings = [get_embedding(doc) for doc in documents]

# ---- Your search query ----
query = "Score-based generative models and denoising score matching"

print(f"\n🔍 Query: '{query}'")
query_embedding = get_embedding(query)

# ---- Compute similarities ----
results = []
for i, doc_emb in enumerate(doc_embeddings):
    score = cosine_similarity(query_embedding, doc_emb)
    results.append((score, documents[i]))

# ---- Rank and display ----
results.sort(reverse=True)

print("\n📊 Results (ranked by similarity):")
print("-" * 60)
for score, doc in results:
    bar = "█" * int(score * 20)
    print(f"Score: {score:.4f} {bar}")
    print(f"Text:  {doc}\n")

📦 Embedding documents...

🔍 Query: 'Score-based generative models and denoising score matching'

📊 Results (ranked by similarity):
------------------------------------------------------------
Score: 0.3803 ███████
Text:  Scale Space Diffusion. Diffusion models degrade images through noise, and reversing this process reveals an information hierarchy across timesteps. Scale-space theory exhibits a similar hierarchy via low-pass filtering. We formalize this connection and show that highly noisy diffusion states contain no more information than small, downsampled images - raising the question of why they must be processed at full resolution. To address this, we fuse scale spaces into the diffusion process by formulating a family of diffusion models with generalized linear degradations and practical implementations. Using downsampling as the degradation yields our proposed Scale Space Diffusion. To support Scale Space Diffusion, we introduce Flexi-UNet, a UNet variant that performs resoluti

In [14]:
import numpy as np
import requests

def get_embedding(text):
    response = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={
            "Authorization": f"Bearer {JINA_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "input": [text],
            "model": "jina-embeddings-v3"  # 1024-dim, multilingual
        }
    )
    return np.array(response.json()["data"][0]["embedding"])

def cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

# ---- Your documents (the "database") ----
documents = [
    text_to_embed,
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language for data science.",
    "Machine learning is a subset of artificial intelligence.",
    "The Great Wall of China is one of the seven wonders.",
    "Neural networks are inspired by the human brain.",
]

# ---- Embed all documents ----
print("📦 Embedding documents with Jina...")
doc_embeddings = [get_embedding(doc) for doc in documents]
print(f"✅ Vector dimension: {len(doc_embeddings[0])}")  # 1024 for jina-embeddings-v3

# ---- Your search query ----
query = "Score-based generative models and denoising score matching"

print(f"\n🔍 Query: '{query}'")
query_embedding = get_embedding(query)

# ---- Compute similarities ----
results = []
for i, doc_emb in enumerate(doc_embeddings):
    score = cosine_similarity(query_embedding, doc_emb)
    results.append((score, documents[i]))

# ---- Rank and display ----
results.sort(reverse=True)

print("\n📊 Results (ranked by similarity):")
print("-" * 60)
for score, doc in results:
    bar = "█" * int(score * 20)
    print(f"Score: {score:.4f} {bar}")
    print(f"Text:  {doc[:80]}{'...' if len(doc) > 80 else ''}\n")

📦 Embedding documents with Jina...
✅ Vector dimension: 1024

🔍 Query: 'Score-based generative models and denoising score matching'

📊 Results (ranked by similarity):
------------------------------------------------------------
Score: 0.4248 ████████
Text:  Scale Space Diffusion. Diffusion models degrade images through noise, and revers...

Score: 0.3601 ███████
Text:  Python is a popular programming language for data science.

Score: 0.3471 ██████
Text:  Neural networks are inspired by the human brain.

Score: 0.2931 █████
Text:  Machine learning is a subset of artificial intelligence.

Score: 0.2471 ████
Text:  The Great Wall of China is one of the seven wonders.

Score: 0.2047 ████
Text:  The Eiffel Tower is located in Paris, France.

